# 10 — Connection Pool (Amazon FAR style)

A pool that hands out a bounded number of reusable connections. Parts 1-3 build the core (concurrency
at Part 3); Parts 4-5 are stretch tasks (timeouts/eviction, then sharding + parallel warmup). Each
part is its own class so they stay independent. Fill stubs, run each test cell, peek at the collapsed
`ref_` solutions only after trying.

A "connection" here is whatever a `factory()` returns (an int id, a string, …); the pool just manages
lifecycle, not real sockets.

---

## Part 1 — Basic pool

`ConnectionPool(factory, max_size)`: `acquire()` returns an idle connection, reusing a released one or
**lazily** creating a new one up to `max_size`; `release(conn)` returns it to the pool. Plus
`created()` (total ever made) and `available()` (idle count).

**Lock down:** reuse before create; never create more than `max_size`; releasing makes a connection
available again. (Single-threaded for now.)

In [ ]:
class ConnectionPool:
    def __init__(self, factory, max_size):
        raise NotImplementedError

    def acquire(self):
        raise NotImplementedError

    def release(self, conn):
        raise NotImplementedError

    def created(self):
        raise NotImplementedError

    def available(self):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    ids = iter(range(1000))
    p = ConnectionPool(lambda: next(ids), max_size=2)
    a = p.acquire(); b = p.acquire()
    assert p.created() == 2 and a != b
    p.release(a)
    c = p.acquire()                                # reuse the released one, no new creation
    assert c == a and p.created() == 2 and p.available() == 0
    print("Part 1 OK")

_t1()

## Part 2 — Validate on acquire

`ValidatingPool(factory, max_size, is_valid)`: same as Part 1, but on `acquire()` an idle connection
that fails `is_valid(conn)` is **discarded** (freeing its slot) and replaced — so callers never get a
dead connection.

**Lock down:** a discarded connection decrements the live count so a fresh one can be created; keep
trying until you return a valid connection.

In [ ]:
class ValidatingPool:
    def __init__(self, factory, max_size, is_valid):
        raise NotImplementedError

    def acquire(self):
        raise NotImplementedError

    def release(self, conn):
        raise NotImplementedError

    def created(self):
        raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    ids = iter(range(1000))
    broken = set()
    p = ValidatingPool(lambda: next(ids), max_size=1, is_valid=lambda c: c not in broken)
    a = p.acquire(); assert a == 0
    p.release(a)
    broken.add(0)                                  # connection 0 has gone bad
    b = p.acquire()                                # 0 discarded, replaced with a fresh one
    assert b == 1 and p.created() == 1
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe blocking pool

`BlockingPool(factory, max_size)`: now thread-safe. When the pool is exhausted, `acquire()` **blocks**
until another thread `release()`s one (no busy-waiting). Adds `in_use()`.

**The invariant:** under any number of concurrent threads, the pool must **never create more than
`max_size`** connections, and every acquire must eventually be served. **Discuss:** `Condition` /
semaphore for blocking + wakeup; the check-then-create race; fairness.

In [ ]:
import threading


class BlockingPool:
    def __init__(self, factory, max_size):
        raise NotImplementedError

    def acquire(self):
        raise NotImplementedError

    def release(self, conn):
        raise NotImplementedError

    def created(self):
        raise NotImplementedError

    def available(self):
        raise NotImplementedError

    def in_use(self):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    ids = iter(range(100000))
    p = BlockingPool(lambda: next(ids), max_size=4)
    M, per = 200, 10

    def worker():
        for _ in range(per):
            c = p.acquire()
            p.release(c)

    ts = [threading.Thread(target=worker) for _ in range(M)]
    for t in ts: t.start()
    for t in ts: t.join()
    assert p.created() <= 4                         # cap never exceeded under contention
    assert p.in_use() == 0 and p.available() == p.created()   # everything returned
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Timeouts, eviction, double-release

`RobustPool(factory, max_size)`:
- `acquire(timeout=None)`: raise `TimeoutError` if no connection becomes available within `timeout`.
- `release(conn, broken=False)`: a `broken=True` connection is **dropped** (frees its slot) instead of
  being reused.
- detect misuse: releasing a connection that isn't currently checked out raises `RuntimeError`.

**Discuss:** health-checks / max-lifetime eviction, leaked-connection detection, and why a fair
(FIFO) wait queue matters under heavy contention.

In [ ]:
class RobustPool:
    def __init__(self, factory, max_size):
        raise NotImplementedError

    def acquire(self, timeout=None):
        raise NotImplementedError

    def release(self, conn, broken=False):
        raise NotImplementedError

    def created(self):
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    ids = iter(range(1000))
    p = RobustPool(lambda: next(ids), max_size=1)
    a = p.acquire()
    try:
        p.acquire(timeout=0.05)                     # exhausted -> times out
    except TimeoutError:
        pass
    else:
        raise AssertionError("expected TimeoutError")
    p.release(a, broken=True)                        # dropped, frees the slot
    assert p.created() == 0
    b = p.acquire()
    assert p.created() == 1
    p.release(b)
    try:
        p.release(b)                                 # already released
    except RuntimeError:
        pass
    else:
        raise AssertionError("expected double-release RuntimeError")
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Sharded pools + parallel warmup

**(a)** `ShardedPool(make, max_size)`: an independent sub-pool **per key** (e.g. per host/db), created
lazily. `acquire(key)` / `release(key, conn)`; `make(key)` builds a connection for that key. Keys don't
share connections.

**(b)** `parallel_warmup(keys, num_procs) -> dict[key, value]`: precompute the expensive
per-key handshake across processes with `ProcessPoolExecutor`; worker is
`connpool_workers.expensive_handshake`.

**Discuss:** one lock per shard vs a global lock, hot-key skew, and overlapping connection warmup
(CPU/crypto-bound) across processes.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import connpool_workers


class ShardedPool:
    def __init__(self, make, max_size):
        raise NotImplementedError

    def acquire(self, key):
        raise NotImplementedError

    def release(self, key, conn):
        raise NotImplementedError


def parallel_warmup(keys, num_procs) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    sp = ShardedPool(make=lambda k: f"{k}-conn", max_size=2)
    a = sp.acquire("db"); b = sp.acquire("cache")
    assert a == "db-conn" and b == "cache-conn"
    sp.release("db", a)
    assert sp.acquire("db") == a                    # reused within the same key
    warmed = parallel_warmup([1, 2, 3, 4], num_procs=2)
    assert warmed == {k: sum(i * i for i in range(k)) for k in (1, 2, 3, 4)}
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Min/max idle, max-lifetime, idle-reaper background thread.
- Fair (FIFO) waiters vs barging; `acquire` cancellation.
- Async pools (`asyncio`), and pool-per-event-loop vs shared.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
import threading
from concurrent.futures import ProcessPoolExecutor
import connpool_workers


class RefConnectionPool:
    def __init__(self, factory, max_size):
        self._factory, self._max = factory, max_size
        self._idle, self._created = [], 0

    def acquire(self):
        if self._idle:
            return self._idle.pop()
        if self._created < self._max:
            self._created += 1
            return self._factory()
        raise RuntimeError("pool exhausted")

    def release(self, conn):
        self._idle.append(conn)

    def created(self):
        return self._created

    def available(self):
        return len(self._idle)


class RefValidatingPool:
    def __init__(self, factory, max_size, is_valid):
        self._factory, self._max, self._is_valid = factory, max_size, is_valid
        self._idle, self._created = [], 0

    def acquire(self):
        while True:
            if self._idle:
                c = self._idle.pop()
                if self._is_valid(c):
                    return c
                self._created -= 1                  # drop the dead one, free a slot
                continue
            if self._created < self._max:
                self._created += 1
                return self._factory()
            raise RuntimeError("pool exhausted")

    def release(self, conn):
        self._idle.append(conn)

    def created(self):
        return self._created


class RefBlockingPool:
    def __init__(self, factory, max_size):
        self._factory, self._max = factory, max_size
        self._idle, self._created, self._in_use = [], 0, 0
        self._cv = threading.Condition()

    def acquire(self):
        with self._cv:
            while True:
                if self._idle:
                    self._in_use += 1
                    return self._idle.pop()
                if self._created < self._max:
                    self._created += 1
                    self._in_use += 1
                    return self._factory()
                self._cv.wait()

    def release(self, conn):
        with self._cv:
            self._in_use -= 1
            self._idle.append(conn)
            self._cv.notify()

    def created(self):
        with self._cv:
            return self._created

    def available(self):
        with self._cv:
            return len(self._idle)

    def in_use(self):
        with self._cv:
            return self._in_use


class RefRobustPool:
    def __init__(self, factory, max_size):
        self._factory, self._max = factory, max_size
        self._idle, self._created = [], 0
        self._checked_out = set()
        self._cv = threading.Condition()

    def acquire(self, timeout=None):
        with self._cv:
            while True:
                if self._idle:
                    c = self._idle.pop()
                    self._checked_out.add(c)
                    return c
                if self._created < self._max:
                    c = self._factory()
                    self._created += 1
                    self._checked_out.add(c)
                    return c
                if not self._cv.wait(timeout):
                    raise TimeoutError("pool exhausted")

    def release(self, conn, broken=False):
        with self._cv:
            if conn not in self._checked_out:
                raise RuntimeError("connection not checked out")
            self._checked_out.discard(conn)
            if broken:
                self._created -= 1                  # drop it
            else:
                self._idle.append(conn)
            self._cv.notify()

    def created(self):
        with self._cv:
            return self._created


class RefShardedPool:
    def __init__(self, make, max_size):
        self._make, self._max = make, max_size
        self._pools, self._lock = {}, threading.Lock()

    def _pool(self, key):
        with self._lock:
            if key not in self._pools:
                self._pools[key] = RefBlockingPool(lambda k=key: self._make(k), self._max)
            return self._pools[key]

    def acquire(self, key):
        return self._pool(key).acquire()

    def release(self, key, conn):
        self._pool(key).release(conn)


def ref_parallel_warmup(keys, num_procs):
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return dict(zip(keys, ex.map(connpool_workers.expensive_handshake, keys)))


_p = RefConnectionPool(iter(range(9)).__next__, 2)
_a = _p.acquire(); _p.release(_a); assert _p.acquire() == _a and _p.created() == 1
_seq = iter(range(9)); _bad = set()
_v = RefValidatingPool(lambda: next(_seq), 1, lambda c: c not in _bad)
_c0 = _v.acquire(); _v.release(_c0); _bad.add(_c0)
assert _v.acquire() == 1                            # 0 went bad while idle, replaced
_r = RefRobustPool(iter(range(9)).__next__, 1); _x = _r.acquire(); _r.release(_x, broken=True)
assert _r.created() == 0
assert ref_parallel_warmup([1, 2], 2) == {1: 0, 2: 1}
print("reference solutions OK")